# Walmart Sales Analysis
**Dataset:** Walmart_Sales.csv — 45 stores, weekly data from 2010 to 2012

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

## Load Data

In [ ]:
df = pd.read_csv('Walmart_Sales.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

## Data Cleaning

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Week']  = df['Date'].dt.isocalendar().week.astype(int)

df.drop_duplicates(inplace=True)

print('Clean shape:', df.shape)
df.dtypes

In [ ]:
df.describe()

## Outlier Check

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].boxplot(df['Weekly_Sales'], vert=False, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7),
                medianprops=dict(color='red', linewidth=2))
axes[0].set_title('Weekly Sales — Boxplot')
axes[0].set_xlabel('Weekly Sales ($)')

axes[1].hist(df['Weekly_Sales'], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_title('Weekly Sales — Distribution')
axes[1].set_xlabel('Weekly Sales ($)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

**Insight:** Sales are right-skewed — most stores do moderate weekly sales but a few high-volume stores pull the average up. These outliers are real, not errors.

---

## EDA — Business Questions

### 1. Which stores generate the highest revenue?

In [ ]:
store_sales = df.groupby('Store')['Weekly_Sales'].sum().sort_values(ascending=False)

plt.figure(figsize=(14, 5))
colors = ['#1565C0' if i < 5 else '#90CAF9' for i in range(len(store_sales))]
store_sales.plot(kind='bar', color=colors, edgecolor='white')
plt.title('Total Revenue by Store (dark = top 5)')
plt.xlabel('Store')
plt.ylabel('Total Sales ($)')
plt.tight_layout()
plt.show()

print('Top 5 stores:')
print(store_sales.head())

**Insight:** The top 5 stores bring in a disproportionate chunk of total revenue. These stores are the backbone — any disruption here has a huge impact on overall numbers.

**Recommendation:** Invest extra in these stores — better staffing, faster restocking, and higher inventory buffers.

### 2. Holiday vs Non-Holiday Sales

In [ ]:
holiday_avg = df.groupby('Holiday_Flag')['Weekly_Sales'].mean()

plt.figure(figsize=(6, 4))
bars = plt.bar(['Regular Week', 'Holiday Week'], holiday_avg.values,
               color=['#90CAF9', '#EF5350'], edgecolor='white', width=0.4)
for bar, val in zip(bars, holiday_avg.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
             f'${val:,.0f}', ha='center', fontweight='bold', fontsize=10)
plt.title('Avg Weekly Sales — Holiday vs Regular')
plt.ylabel('Avg Weekly Sales ($)')
plt.tight_layout()
plt.show()

lift = (holiday_avg[1] / holiday_avg[0] - 1) * 100
print(f'Holiday weeks are {lift:.1f}% higher than regular weeks')

**Insight:** Holiday weeks clearly outperform regular ones. Thanksgiving, Christmas, and Super Bowl weeks see the biggest spikes.

**Recommendation:** Pre-load inventory 2–3 weeks before major holidays. Don't get caught understocked when traffic peaks.

### 3. Monthly Sales Trend

In [ ]:
monthly = df.groupby('Month')['Weekly_Sales'].mean()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

plt.figure(figsize=(11, 4))
plt.plot(monthly.index, monthly.values, marker='o', color='steelblue', linewidth=2)
plt.fill_between(monthly.index, monthly.values, alpha=0.15, color='steelblue')
plt.xticks(range(1, 13), month_labels)
plt.title('Average Weekly Sales by Month')
plt.xlabel('Month')
plt.ylabel('Avg Weekly Sales ($)')
plt.tight_layout()
plt.show()

**Insight:** Two clear peaks — Nov/Dec (holiday season) and a smaller bump in July/August (back-to-school). January is the lowest month — post-holiday spending hangover.

**Recommendation:** Q4 is the make-or-break quarter. Supply chain prep should start by September at the latest.

### 4. Yearly Sales Trend

In [ ]:
yearly = df.groupby('Year')['Weekly_Sales'].sum() / 1e9

plt.figure(figsize=(6, 4))
bars = plt.bar(yearly.index.astype(str), yearly.values,
               color=['#42A5F5','#1565C0','#0D47A1'], edgecolor='white', width=0.4)
for bar, val in zip(bars, yearly.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'${val:.2f}B', ha='center', fontweight='bold')
plt.title('Total Revenue by Year')
plt.ylabel('Total Sales ($ Billions)')
plt.tight_layout()
plt.show()

**Insight:** Revenue stays relatively stable year over year — this is a mature, consistent business. Not explosive growth, but very predictable.

**Recommendation:** The predictability here is a superpower for supply chain planning. Invest in forecasting tools — the ROI is clear.

### 5. Sales Over Time (All Stores)

In [ ]:
time_trend = df.groupby('Date')['Weekly_Sales'].sum() / 1e6

plt.figure(figsize=(14, 4))
plt.plot(time_trend.index, time_trend.values, color='steelblue', linewidth=1.2)
plt.fill_between(time_trend.index, time_trend.values, alpha=0.15, color='steelblue')

holiday_weeks = df[df['Holiday_Flag'] == 1]['Date'].unique()
for hd in holiday_weeks:
    plt.axvline(x=hd, color='red', alpha=0.15, linewidth=0.8)

plt.title('Total Weekly Sales Over Time (red lines = holiday weeks)')
plt.xlabel('Date')
plt.ylabel('Total Sales ($ Millions)')
plt.tight_layout()
plt.show()

**Insight:** The same seasonal pattern repeats every year like clockwork. Holiday spikes are sharp and short — they demand fast, precise execution.

**Recommendation:** This repeating pattern is perfect for time-series forecasting models (Prophet, SARIMA, or XGBoost with lag features).

### 6. Temperature Impact on Sales

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['Temperature'], df['Weekly_Sales'], alpha=0.2, color='steelblue', s=10)
z = np.polyfit(df['Temperature'], df['Weekly_Sales'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['Temperature'].min(), df['Temperature'].max(), 100)
plt.plot(x_line, p(x_line), color='red', linewidth=2, label='Trend')
plt.title('Temperature vs Weekly Sales')
plt.xlabel('Temperature (°F)')
plt.ylabel('Weekly Sales ($)')
plt.legend()
plt.tight_layout()
plt.show()

print('Correlation:', round(df['Temperature'].corr(df['Weekly_Sales']), 3))

**Insight:** Very weak correlation. Temperature alone doesn't drive Walmart sales much — but seasonal product categories (fans, heaters, cold drinks) are definitely temperature-sensitive.

**Recommendation:** Category-level temperature planning matters more than store-level. Seasonal product buyers should watch local forecasts.

### 7. Fuel Price Impact on Sales

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['Fuel_Price'], df['Weekly_Sales'], alpha=0.2, color='orange', s=10)
z = np.polyfit(df['Fuel_Price'], df['Weekly_Sales'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['Fuel_Price'].min(), df['Fuel_Price'].max(), 100)
plt.plot(x_line, p(x_line), color='red', linewidth=2, label='Trend')
plt.title('Fuel Price vs Weekly Sales')
plt.xlabel('Fuel Price ($/gallon)')
plt.ylabel('Weekly Sales ($)')
plt.legend()
plt.tight_layout()
plt.show()

print('Correlation:', round(df['Fuel_Price'].corr(df['Weekly_Sales']), 3))

**Insight:** Mild negative trend — when fuel gets expensive, customers may consolidate shopping trips or shop online more. Walmart's one-stop format helps absorb this.

**Recommendation:** During fuel price spikes, push click-and-collect and grocery pickup services harder.

### 8. CPI Impact on Sales

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['CPI'], df['Weekly_Sales'], alpha=0.2, color='purple', s=10)
z = np.polyfit(df['CPI'], df['Weekly_Sales'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['CPI'].min(), df['CPI'].max(), 100)
plt.plot(x_line, p(x_line), color='red', linewidth=2, label='Trend')
plt.title('CPI vs Weekly Sales')
plt.xlabel('Consumer Price Index')
plt.ylabel('Weekly Sales ($)')
plt.legend()
plt.tight_layout()
plt.show()

print('Correlation:', round(df['CPI'].corr(df['Weekly_Sales']), 3))

**Insight:** Higher CPI (inflation) shows a mild positive relationship with Walmart sales. When prices rise elsewhere, shoppers trade down to Walmart for value — the classic "trading down" effect.

**Recommendation:** During inflationary periods, lean hard into the "Save Money. Live Better." positioning. Private label products should get more shelf space.

### 9. Unemployment Impact on Sales

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['Unemployment'], df['Weekly_Sales'], alpha=0.2, color='red', s=10)
z = np.polyfit(df['Unemployment'], df['Weekly_Sales'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['Unemployment'].min(), df['Unemployment'].max(), 100)
plt.plot(x_line, p(x_line), color='darkred', linewidth=2, label='Trend')
plt.title('Unemployment Rate vs Weekly Sales')
plt.xlabel('Unemployment Rate (%)')
plt.ylabel('Weekly Sales ($)')
plt.legend()
plt.tight_layout()
plt.show()

print('Correlation:', round(df['Unemployment'].corr(df['Weekly_Sales']), 3))

**Insight:** Higher unemployment has a mild negative effect on sales. But Walmart's low-price model acts as a cushion — people still need essentials regardless of economic conditions.

**Recommendation:** In high-unemployment markets, shift the product mix toward essentials and value packs. Cut back on premium/discretionary promotions.

### 10. Correlation Heatmap

In [ ]:
cols = ['Weekly_Sales', 'Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
corr = df[cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, square=True)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

**Insight:** `Holiday_Flag` has the strongest correlation with `Weekly_Sales`. CPI and Unemployment are negatively correlated with each other (typical recession pattern). No feature alone is a silver bullet — a model needs all of them together.

---

## Save Cleaned Dataset

In [ ]:
df.to_csv('cleaned_walmart_sales.csv', index=False)
print('Saved: cleaned_walmart_sales.csv')
print('Shape:', df.shape)

---

## Summary

| Question | Finding |
|----------|---------|
| Top stores | A few stores dominate total revenue |
| Holiday impact | Holiday weeks are noticeably higher |
| Seasonal trend | Nov–Dec peak, January dip every year |
| Yearly trend | Stable and predictable revenue |
| Temperature | Weak effect overall |
| Fuel price | Mild negative impact on sales |
| CPI | Higher inflation slightly helps Walmart |
| Unemployment | Mild negative, but Walmart is resilient |

---

**Next step:** Feature engineering + sales forecasting model